# Python 基础：从语法到可复用代码

**学习目标**

1. 理解变量、对象、可变性和常用容器；
2. 能用条件、循环、函数和异常处理组织程序；
3. 能读懂类、推导式、迭代器等常见 Python 写法；
4. 养成“小函数、清晰命名、主动验证”的编码习惯。

> 建议：按顺序运行代码单元。每节先预测输出，再执行验证。


## 0. 从 C++ 迁移到 Python：先建立语言地图

你已有 C++ 基础，因此很多控制流和面向对象概念并不陌生。主要差异在语法、对象模型和常用编程习惯。

| C++ | Python | 需要注意 |
|---|---|---|
| `{ ... }` 表示代码块 | 缩进表示代码块 | 推荐每层 4 个空格 |
| 语句通常以 `;` 结束 | 换行结束语句 | 通常不写分号 |
| `int x = 1;` | `x = 1` | 名称没有固定静态类型 |
| `std::vector<T>` | `list` | list 可混合类型，但实际项目不建议乱混 |
| `std::map` / `unordered_map` | `dict` | 键值映射 |
| `std::set` | `set` | 去重和集合运算 |
| `nullptr` | `None` | 使用 `is None` 判断 |
| `for (int i=0; ...)` | `for value in iterable` | 优先直接遍历元素 |
| 指针 / 引用 | 名称绑定到对象 | 可变对象共享引用时要特别小心 |
| 模板类型提示 | type hints | Python 类型标注通常不强制运行时检查 |

### 最重要的思维转换

Python 变量不是装有值的固定类型盒子，而是指向对象的名称：

```text
名称 x ─────► 某个对象
```

`x = another_object` 是让名称重新指向另一个对象。对可变对象的修改则会被所有共享该对象的名称看到。


In [1]:
values = [10, 20, 30]

# C++ 常用索引循环；Python 更常直接遍历元素。
for value in values:
    print("value:", value)

# 同时需要索引时使用 enumerate。
for index, value in enumerate(values):
    print(index, value)

# 同时遍历两个序列时使用 zip。
names = ["Alice", "Bob", "Chen"]
for name, score in zip(names, values):
    print(f"{name}: {score}")


value: 10
value: 20
value: 30
0 10
1 20
2 30
Alice: 10
Bob: 20
Chen: 30


### Python 函数与 C++ 函数的对应

Python 不写返回类型和参数类型也能运行，但类型标注可以提高可读性：

```python
def add(left: int, right: int) -> int:
    return left + right
```

与 C++ 不同，Python 函数可以返回多个值；本质上返回的是一个 tuple：

```python
minimum, maximum = find_range(values)
```

初学阶段建议始终明确写 `return`，并让一个函数只完成一件事。


## 1. 变量、对象与类型

Python 变量更像“贴在对象上的标签”，而不是固定类型的盒子。  
`=` 是绑定名称，`==` 比较值，`is` 比较是否为同一个对象。


In [2]:
age = 20
price = 19.9
name = "Jason"
is_student = True
nothing = None

for value in [age, price, name, is_student, nothing]:
    print(f"{value!r:>10} -> {type(value).__name__}")

# 判断空值使用 is None；比较普通值使用 ==
result = None
print("尚无结果：", result is None)


        20 -> int
      19.9 -> float
   'Jason' -> str
      True -> bool
      None -> NoneType
尚无结果： True


### 可变对象与不可变对象

- 不可变：`int`、`float`、`bool`、`str`、`tuple`
- 可变：`list`、`dict`、`set`

多个变量指向同一个可变对象时，通过其中一个变量修改，其他变量也会看到变化。


In [3]:
original = [1, 2, 3]
alias = original          # 指向同一个列表
copied = original.copy()  # 新建一个浅拷贝

alias.append(4)
print("original:", original)
print("alias is original:", alias is original)
print("copied:", copied)


original: [1, 2, 3, 4]
alias is original: True
copied: [1, 2, 3]


### 数字类型补全：`int`、`float`、`complex` 与 `bool`

- `int`：整数，Python 会自动扩展大小；
- `float`：双精度浮点数，适合一般科学计算，但不能精确表示所有十进制小数；
- `complex`：复数，写作 `a + bj`；
- `bool`：`True` / `False`，用于条件判断。

浮点数使用二进制存储，所以 `0.1 + 0.2` 不一定精确等于 `0.3`。比较计算结果时可使用 `math.isclose()`。


In [4]:
import math

integer_value = 42
float_value = 3.14
complex_value = 2 + 3j
boolean_value = True

print(type(integer_value), type(float_value))
print(type(complex_value), complex_value.real, complex_value.imag)
print(type(boolean_value))

print("0.1 + 0.2 =", 0.1 + 0.2)
print("接近 0.3:", math.isclose(0.1 + 0.2, 0.3))


<class 'int'> <class 'float'>
<class 'complex'> 2.0 3.0
<class 'bool'>
0.1 + 0.2 = 0.30000000000000004
接近 0.3: True


### `==`、`is` 与 `del`

- `==` 比较两个对象的**值**；
- `is` 比较是否为**同一个对象**；
- `del name` 删除的是名称绑定，不保证立即销毁对象。

判断空值使用 `value is None`；比较数字、字符串和容器内容通常使用 `==`。


In [5]:
first = [1, 2]
second = [1, 2]
alias = first

print("first == second:", first == second)
print("first is second:", first is second)
print("first is alias:", first is alias)

temporary = "仍可能被其他引用持有的对象"
del temporary
# 再访问 temporary 会触发 NameError，因为这个名称已被删除。


first == second: True
first is second: False
first is alias: True


## 2. 字符串、格式化与切片

切片统一写作 `序列[start:stop:step]`，范围**左闭右开**。负索引从末尾计数。


In [6]:
text = "python"
print(text[0], text[-1])       # 首字符、尾字符
print(text[1:4])               # yth
print(text[::-1])              # 反转

user = "Jason"
score = 92.345
print(f"{user} 的成绩是 {score:.1f}")  # f-string：保留 1 位小数

raw_path = r"C:\new_folder"    # r 前缀让反斜杠按原样保留
print(raw_path)


p n
yth
nohtyp
Jason 的成绩是 92.3
C:\new_folder


### 输出控制、用户输入与长表达式

- `print(..., end="")` 控制结尾字符；
- `input(prompt)` 返回字符串，需要按业务转换类型；
- 长表达式优先放在括号中自然换行；
- 反斜杠也能续行，但更容易因空格导致错误；
- 分号可以把多条语句放在一行，但不推荐。

为避免 Notebook 执行时暂停，下面只展示 `input` 的标准写法，不实际等待输入。

```python
raw_age = input("请输入年龄：")
age = int(raw_age)
```


In [7]:
print("A", end=" -> ")
print("B")

total = (
    10
    + 20
    + 30
)
print("括号内换行的结果:", total)


A -> B
括号内换行的结果: 60


## 3. 运算符与流程控制

比较可以连写：`0 <= x < 10`。  
`and`、`or`、`not` 用于组合条件；空容器、`0`、`None` 会被视为假。


In [8]:
temperature = 26

if temperature >= 30:
    advice = "炎热"
elif temperature >= 20:
    advice = "舒适"
else:
    advice = "偏冷"

print(advice)


舒适


In [9]:
numbers = [1, 2, 3, 4, 5]

total = 0
for number in numbers:
    if number % 2 == 0:
        continue             # 跳过偶数
    total += number

print("奇数之和:", total)

countdown = 3
while countdown > 0:
    print(countdown)
    countdown -= 1


奇数之和: 9
3
2
1


## 4. 四种核心容器

| 类型 | 是否有序 | 是否可变 | 是否去重 | 典型用途 |
|---|---:|---:|---:|---|
| `list` | 是 | 是 | 否 | 一组按顺序存放的数据 |
| `tuple` | 是 | 否 | 否 | 不希望被修改的记录 |
| `dict` | 保持插入顺序 | 是 | 键唯一 | 键值映射、结构化记录 |
| `set` | 不保证业务顺序 | 是 | 是 | 去重、集合运算 |


In [10]:
fruits = ["apple", "banana", "apple"]
point = (3, 4)
student = {"name": "Jason", "score": 92}
unique_fruits = set(fruits)

student["passed"] = student["score"] >= 60
print(student)
print("去重:", unique_fruits)
print("交集:", {1, 2, 3} & {2, 3, 4})


{'name': 'Jason', 'score': 92, 'passed': True}
去重: {'banana', 'apple'}
交集: {2, 3}


### 推导式：把“筛选 + 变换”写清楚

推导式适合简单逻辑；逻辑超过一层时，普通循环通常更易读。


In [11]:
numbers = range(10)
even_squares = [n ** 2 for n in numbers if n % 2 == 0]
square_map = {n: n ** 2 for n in range(4)}

print(even_squares)
print(square_map)


[0, 4, 16, 36, 64]
{0: 0, 1: 1, 2: 4, 3: 9}


## 5. 函数：输入、处理、输出

函数应尽量只做一件事。参数负责输入，`return` 负责输出；不要依赖难以追踪的全局变量。


In [12]:
def calculate_mean(values: list[float]) -> float:
    """返回非空数值序列的平均值。"""
    if not values:
        raise ValueError("values 不能为空")
    return sum(values) / len(values)


scores = [88, 92, 79]
average = calculate_mean(scores)
print(f"平均分: {average:.2f}")


平均分: 86.33


In [13]:
def describe(name: str, score: float = 60, *, precision: int = 1) -> str:
    """星号后的参数必须用关键字传入，调用含义更明确。"""
    return f"{name}: {score:.{precision}f}"


print(describe("Jason", 92.345, precision=2))

# *args 收集位置参数；**kwargs 收集关键字参数
def summarize(*values: float, **labels: str) -> dict:
    return {"total": sum(values), "count": len(values), **labels}


print(summarize(1, 2, 3, unit="points"))


Jason: 92.34
{'total': 6, 'count': 3, 'unit': 'points'}


## 6. 异常处理：只捕获你能处理的错误

不要用裸 `except:` 隐藏所有错误。优先捕获具体异常，并给出可行动的信息。


In [14]:
def parse_age(text: str) -> int | None:
    try:
        age = int(text)
        if age < 0:
            raise ValueError("年龄不能为负数")
        return age
    except ValueError as error:
        print(f"输入无效：{error}")
        return None


print(parse_age("20"))
print(parse_age("twenty"))


20
输入无效：invalid literal for int() with base 10: 'twenty'
None


## 7. 类、数据类与对象

当数据和操作天然属于同一个概念时，可以使用类。只保存数据时，`dataclass` 能减少模板代码。


In [15]:
from dataclasses import dataclass


@dataclass
class Student:
    name: str
    scores: list[float]

    def average(self) -> float:
        return sum(self.scores) / len(self.scores)


student = Student("Jason", [88, 92, 79])
print(student)
print(f"平均分: {student.average():.1f}")


Student(name='Jason', scores=[88, 92, 79])
平均分: 86.3


## 8. 迭代器与生成器

生成器用 `yield` 按需产生值，不会一次把全部结果放入内存，适合大数据流。


In [16]:
def countdown(start: int):
    current = start
    while current > 0:
        yield current
        current -= 1


for value in countdown(3):
    print(value)


3
2
1


## 9. 常见坑与自测

- 不要把 `list`、`str`、`sum` 等内置名称当作变量名。
- 函数默认参数不要使用可变对象，如 `def f(items=[])`。
- 金额等精确小数不宜直接依赖二进制浮点数。
- 代码块依靠缩进；推荐每层 4 个空格。

**练习**

1. 写 `normalize_name(text)`：去掉两端空格，并把每个单词首字母大写。
2. 写 `word_count(sentence)`：返回每个单词出现次数的字典。
3. 写 `safe_divide(a, b)`：除数为 0 时返回 `None`。


In [17]:
# 参考答案：先自己完成，再展开对照
def normalize_name(text: str) -> str:
    return text.strip().title()


def word_count(sentence: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    for word in sentence.lower().split():
        counts[word] = counts.get(word, 0) + 1
    return counts


def safe_divide(a: float, b: float) -> float | None:
    return None if b == 0 else a / b


print(normalize_name("  jason lee "))
print(word_count("to be or not to be"))
print(safe_divide(8, 2), safe_divide(8, 0))


Jason Lee
{'to': 2, 'be': 2, 'or': 1, 'not': 1}
4.0 None
